# DFDC Training By Isit Thakkar(1002229820)
# Krushna Bhujbal(1002241902)

This notebook trains the VisionKAN + VisionLSTM combined model on the DFDCDFDC Kaggle dataset **using CPU**.



In [1]:
# Imports
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# Use CPU on MacBook (change to 'cuda' if you have a GPU)
device = torch.device('cpu')
print('Using device:', device)

# If you get import errors, make sure you installed the dependencies, for example:
# pip install torch torchvision torchaudio timm VisionKAN kagglehub pillow matplotlib numpy seaborn scikit-learn tqdm


Using device: cpu


In [2]:
import kagglehub

# Download latest version of DFDCDFDC dataset
path = kagglehub.dataset_download('aleksandrpikul222/dfdcdfdc')
print('Downloaded dataset to:', path)

# Base directory where the txt files live
directory = os.path.join(path, '')  # ensure trailing slash semantics
print('Using directory:', directory)

print('Directory contents:', os.listdir(directory))


Downloaded dataset to: /Users/isitthakkar/.cache/kagglehub/datasets/aleksandrpikul222/dfdcdfdc/versions/3
Using directory: /Users/isitthakkar/.cache/kagglehub/datasets/aleksandrpikul222/dfdcdfdc/versions/3/
Directory contents: ['TEST_IMPOSTER.txt', 'TRAIN_CLIENT.txt', 'VAL_IMPOSTER.txt', 'TEST_CLIENT.txt', 'TRAIN_IMPOSTER.txt', 'VAL_CLIENT.txt', 'DFDCDFDC']


In [ ]:
# Fix absolute paths inside TRAIN_*/TEST_*/VAL_* txt files to match this machine

label_files = [
    'TRAIN_CLIENT.txt',
    'TRAIN_IMPOSTER.txt',
    'VAL_CLIENT.txt',
    'VAL_IMPOSTER.txt',
    'TEST_CLIENT.txt',
    'TEST_IMPOSTER.txt',
]

old_paths = [
    '/kaggle/input/dfdcdfdc/DFDCDFDC/DFDCDFDC/',
    'C://Users/rajat/.cache/kagglehub/datasets/aleksandrpikul222/dfdcdfdc/versions/3/DFDCDFDC/DFDCDFDC/',
]

# New base path for this machine; many txt files store full paths directly
new_base = directory.rstrip('/')
print('New base path to inject into txt files:', new_base)

for fname in label_files:
    file_path = os.path.join(directory, fname)
    if not os.path.exists(file_path):
        print(f'WARNING: {file_path} not found, skipping')
        continue
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        line_stripped = line.strip('\n')
        # Replace any known old base paths with the new one
        for op in old_paths:
            if op in line_stripped:
                line_stripped = line_stripped.replace(op, new_base + '/')
        new_lines.append(line_stripped + '\n')

    with open(file_path, 'w') as f:
        f.writelines(new_lines)
    print(f'Updated paths in {fname}')

print('All label files processed.')


New base path to inject into txt files: /Users/isitthakkar/.cache/kagglehub/datasets/aleksandrpikul222/dfdcdfdc/versions/3
Updated paths in TRAIN_CLIENT.txt
Updated paths in TRAIN_IMPOSTER.txt
Updated paths in VAL_CLIENT.txt
Updated paths in VAL_IMPOSTER.txt
Updated paths in TEST_CLIENT.txt
Updated paths in TEST_IMPOSTER.txt
All label files processed.


In [7]:
# Dataset implementation with manual preprocessing (no torchvision.ToTensor)

def preprocess_pil(img: Image.Image) -> torch.Tensor:
    """
    Resize to 224x224, convert to float tensor, normalize with ImageNet stats.
    """
    # Resize
    img = img.resize((224, 224))

    # To float32 numpy in [0, 1]
    img_np = np.asarray(img, dtype=np.float32) / 255.0

    # Normalize (ImageNet mean/std)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)
    img_np = (img_np - mean) / std

    # HWC -> CHW
    img_np = img_np.transpose(2, 0, 1)

    # To torch tensor
    return torch.tensor(img_np, dtype=torch.float32)


class ASDataset(Dataset):
    """Anti-spoofing dataset with manual preprocessing."""

    def __init__(self, client_file, imposter_file):
        self.samples = []

        # Real = 1, Fake = 0
        for path_file, label in [(client_file, 1), (imposter_file, 0)]:
            with open(path_file, 'r') as f:
                for line in f:
                    p = line.strip()
                    if not p:
                        continue
                    self.samples.append((p, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        # If path is relative, make it absolute using directory
        if not os.path.isabs(img_path):
            img_path_full = os.path.join(directory, img_path)
        else:
            img_path_full = img_path

        img = Image.open(img_path_full).convert('RGB')

        # Manual preprocessing → tensor
        img_tensor = preprocess_pil(img)

        return img_tensor, torch.tensor(label, dtype=torch.float32)


In [9]:
# Create train/val/test datasets (we do preprocessing inside ASDataset)
train_client_file = os.path.join(directory, 'TRAIN_CLIENT.txt')
train_imposter_file = os.path.join(directory, 'TRAIN_IMPOSTER.txt')
val_client_file = os.path.join(directory, 'VAL_CLIENT.txt')
val_imposter_file = os.path.join(directory, 'VAL_IMPOSTER.txt')
test_client_file = os.path.join(directory, 'TEST_CLIENT.txt')
test_imposter_file = os.path.join(directory, 'TEST_IMPOSTER.txt')

train_dataset = ASDataset(train_client_file, train_imposter_file)
val_dataset   = ASDataset(val_client_file,   val_imposter_file)
test_dataset  = ASDataset(test_client_file,  test_imposter_file)

print('Train samples:', len(train_dataset))
print('Val samples:  ', len(val_dataset))
print('Test samples: ', len(test_dataset))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# Sanity check
x_batch, y_batch = next(iter(train_loader))
print('Sample batch shape:', x_batch.shape, y_batch.shape)
print('Sample labels:', y_batch[:10])


Train samples: 80000
Val samples:   10054
Test samples:  9946
Sample batch shape: torch.Size([32, 3, 224, 224]) torch.Size([32])
Sample labels: tensor([0., 0., 0., 1., 0., 0., 1., 1., 1., 0.])


In [10]:
from VisionKAN import create_model

# Build VisionKAN backbone
KAN = create_model(
    model_name='deit_tiny_patch16_224_KAN',
    pretrained=False,
    hdim_kan=192,
    num_classes=1,
    drop_rate=0.0,
    drop_path_rate=0.05,
    img_size=224,
    batch_size=batch_size,
)

print('KAN created.')

# Build VisionLSTM backbone
vision_lstm = torch.hub.load(
    'nx-ai/vision-lstm',
    'VisionLSTM',
    dim=192,
    depth=24,
    patch_size=16,
    input_shape=(3, 224, 224),
    output_shape=(1,),
    drop_path_rate=0.05,
    stride=None,
)
print('VisionLSTM created.')


/opt/anaconda3/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/opt/anaconda3/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


{'hdim_kan': 192, 'num_classes': 1, 'drop_rate': 0.0, 'drop_path_rate': 0.05, 'img_size': 224, 'batch_size': 32}
KAN created.
VisionLSTM created.


Using cache found in /Users/isitthakkar/.cache/torch/hub/nx-ai_vision-lstm_main
/Users/isitthakkar/.cache/torch/hub/nx-ai_vision-lstm_main/vision_lstm/vision_lstm.py:524: UserWarning: You are using an old version of VisionLSTM that uses (i) bilateral_avg pooling instead of bilateral_concat (ii) causal conv1d instead of conv2d before q and k (iii) no biases in projection and layernorms. These three changes improve ImageNet-1K accuracy of a ViL-T from 77.3% to 78.1%. We recommend to use VisionLSTM2 instead of VisionLSTM.
  warnings.warn(


In [18]:
# Combined model definition
class CombinedModel(nn.Module):
    def __init__(self, KAN, vision_lstm):
        super().__init__()
        self.KAN = KAN
        self.vision_lstm = vision_lstm
        self.fc1 = nn.Linear(2, 512)
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        kan_out = self.KAN(x)          # [B, 1]
        lstm_out = self.vision_lstm(x) # [B, 1]
        combined = torch.cat((kan_out, lstm_out), dim=1)  # [B, 2]
        x = torch.relu(self.fc1(combined))
        x = self.fc2(x)  # raw logits
        return x

model = CombinedModel(KAN, vision_lstm).to(device)
print(model)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

num_epochs = 2  # increase to 10+ for better accuracy if you have time


CombinedModel(
  (KAN): VisionKAN(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): kanBlock(
        (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=False)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (act): GELU(approximate='none')
      

In [19]:
from tqdm import tqdm
from collections import OrderedDict

def get_accuracy(preds, labels):
    return torch.sum((preds.flatten() > 0.5) == labels.to(device)) / labels.numel()

msg = OrderedDict({"train_epoch": None, "loss": None, "acc": None,
                   "val_loss": None, "val_acc": None})

def train(epoch):
    running_loss = 0.0
    running_acc = 0.0
    model.train()
    
    for i, (inputs, labels) in (pbar := tqdm(enumerate(train_loader), total=len(train_loader))):
        pbar.set_postfix(**msg)

        optimizer.zero_grad()
        outputs = model(inputs.to(device))
        loss = criterion(outputs, labels.unsqueeze(-1).to(device))
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += get_accuracy(outputs, labels.to(device))

        if not i % 10 and i > 0:
            msg.update({
                "train_epoch": epoch + 1,
                "loss": running_loss / i,
                "acc": (running_acc / i).item()
            })
            pbar.set_postfix(**msg)

def validate():
    val_loss = 0.0
    val_acc = 0.0
    model.eval()

    for i, (inputs, labels) in enumerate(test_loader):
        with torch.no_grad():
            outputs = model(inputs.to(device))

        loss = criterion(outputs, labels.unsqueeze(-1).to(device))
        val_loss += loss.item()
        val_acc += get_accuracy(outputs, labels.to(device))

    val_loss /= len(test_loader)
    val_acc /= len(test_loader)

    scheduler.step()

    msg.update({"val_loss": val_loss, "val_acc": val_acc.item()})
    print(f"val_loss: {val_loss}, val_acc: {val_acc.item()}")

    return val_acc.item()


In [20]:
best_val_acc = 0.0
history = []

for epoch in range(num_epochs):
    train_loss, train_acc = None, None  
    train(epoch)                        

    val_acc = validate()                

    history.append((None, None, None, val_acc))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "as_model_best.pt")
        print(f"  ✅ Saved new best model with val_acc={best_val_acc:.4f} to as_model_best.pt")

print("Training finished. Best validation accuracy:", best_val_acc)


100%|██████████| 2500/2500 [6:00:35<00:00,  8.65s/it, acc=0.856, loss=0.312, train_epoch=1, val_acc=None, val_loss=None]  


val_loss: 0.22253077853470293, val_acc: 0.9401125311851501
  ✅ Saved new best model with val_acc=0.9401 to as_model_best.pt


100%|██████████| 2500/2500 [6:01:01<00:00,  8.66s/it, acc=0.898, loss=0.227, train_epoch=2, val_acc=0.94, val_loss=0.223]  


val_loss: 0.1757609213843595, val_acc: 0.9490554928779602
  ✅ Saved new best model with val_acc=0.9491 to as_model_best.pt
Training finished. Best validation accuracy: 0.9490554928779602


In [21]:
# Test-set evaluation with best model
if os.path.exists('as_model_best.pt'):
    model.load_state_dict(torch.load('as_model_best.pt', map_location=device))
    model.to(device)
    model.eval()
    print('Loaded best model from as_model_best.pt')
else:
    print('WARNING: as_model_best.pt not found; using last model state.')

all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).unsqueeze(1)
        outputs = model(inputs)
        probs = torch.sigmoid(outputs)
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

if all_labels:
    all_labels = np.vstack(all_labels)
    all_probs = np.vstack(all_probs)
    auc = roc_auc_score(all_labels, all_probs)
    print('Test ROC-AUC:', float(auc))
else:
    print('No test data evaluated.')


Loaded best model from as_model_best.pt
Test ROC-AUC: 0.9920684475918651
